# PIP-02 — source dataset audit

Read-only verification of the bundled source, run on 2026-11-08. This notebook
does not clean, split, fit, or export data. See [the audit report](../reports/source_dataset_audit.md)
for the dictionary, retention policy, source references and acceptance evidence.

Run all cells in order with the project environment, from the repository root or
`notebooks/`. The report also provides a terminal rerun without Jupyter.
Assertions intentionally fail if the audited source or its expected properties change.
These full-source descriptions are provenance checks; modeling decisions and
feature exploration must use the training partition in later issues.


In [1]:
import csv
import hashlib
import io
import json
import platform
from pathlib import Path

import pandas as pd

ROOT = next((p for p in (Path.cwd(), Path.cwd().parent)
             if (p / "data/raw/online_shoppers_intention.csv").is_file()), None)
assert ROOT is not None, "Run from the repository root or notebooks/."
source = ROOT / "data/raw/online_shoppers_intention.csv"
source_bytes = source.read_bytes()
expected_sha256 = "b3055ee355f59134d851d32641183cb4a8b45def7124d2f50442a042f358e0d9"
source_sha256 = hashlib.sha256(source_bytes).hexdigest()
assert source_sha256 == expected_sha256, "Source checksum changed; investigate before proceeding."
data = pd.read_csv(io.BytesIO(source_bytes))
print(f"Python {platform.python_version()}; pandas {pd.__version__}")
print(f"Source: {source.relative_to(ROOT)}")
print(f"SHA-256: {source_sha256}")


Python 3.14.8; pandas 3.1.1
Source: data/raw/online_shoppers_intention.csv
SHA-256: b3055ee355f59134d851d32641183cb4a8b45def7124d2f50442a042f358e0d9


## Schema, target and completeness

Integer category codes are nominal labels, not quantities. `Weekend` is a
Boolean predictor; `Revenue` is the Boolean purchase target. Duration fields
contain fractional values despite some UCI table type labels.


In [2]:
numeric_columns = [
    "Administrative", "Administrative_Duration", "Informational",
    "Informational_Duration", "ProductRelated", "ProductRelated_Duration",
    "BounceRates", "ExitRates", "PageValues", "SpecialDay",
]
categorical_columns = ["Month", "OperatingSystems", "Browser", "Region", "TrafficType", "VisitorType"]
expected_columns = numeric_columns + categorical_columns + ["Weekend", "Revenue"]
assert list(data.columns) == expected_columns
assert data.shape == (12330, 18)
assert not data.isna().any().any()
raw_rows = list(csv.reader(io.StringIO(source_bytes.decode("utf-8"))))
assert all(len(row) == 18 for row in raw_rows)
blank_cells = sum(not value.strip() for row in raw_rows[1:] for value in row)
assert blank_cells == 0
assert data["Revenue"].dtype == bool and data["Weekend"].dtype == bool
assert data["Revenue"].value_counts().to_dict() == {False: 10422, True: 1908}
print(f"Shape: {len(data)} rows; {len(data.columns) - 1} predictors + Revenue")
print(f"Missing cells: {int(data.isna().sum().sum())}; blank/whitespace cells: {blank_cells}")
print(data["Revenue"].value_counts().rename_axis("Revenue").to_string())
print(f"Purchase prevalence: {data['Revenue'].mean():.6%}")
print("\nParsed dtypes (semantic roles are in the report):")
print(data.dtypes.to_string())


Shape: 12330 rows; 17 predictors + Revenue
Missing cells: 0; blank/whitespace cells: 0
Revenue
False    10422
True      1908
Purchase prevalence: 15.474453%

Parsed dtypes (semantic roles are in the report):
Administrative               int64
Administrative_Duration    float64
Informational                int64
Informational_Duration     float64
ProductRelated               int64
ProductRelated_Duration    float64
BounceRates                float64
ExitRates                  float64
PageValues                 float64
SpecialDay                 float64
Month                          str
OperatingSystems             int64
Browser                      int64
Region                       int64
TrafficType                  int64
VisitorType                    str
Weekend                       bool
Revenue                       bool


## Repeated rows and predictor-only split groups

Keep all records. Equality of aggregate session summaries is not evidence of
duplicate users. No user identifier is available to establish identity.

Grouping contract `predictor-json-sha256-v1`: use the exact 17 predictor names
in source header order and their pandas-parsed Python scalar values. Serialize
`{"columns": [...], "values": [...]}` with `json.dumps`, `ensure_ascii=False`,
`allow_nan=False`, separators `(',', ':')`, then SHA-256 the UTF-8 bytes.
Exclude `Revenue`, the dataframe index and row position. Equal numeric spellings
that parse to the same scalar are grouped together. Reuse this contract and
parser environment in PIP-03; a changed schema, parser or normalization requires
explicit review. Verify hashes against exact predictor tuples to detect a collision.

PIP-03 must assign a whole group to one partition, including validation folds,
then verify disjoint group sets. These are summary-equality groups, not user IDs.
The notebook defines them in memory; no split or manifest is produced here.


In [3]:
predictor_columns = [column for column in data.columns if column != "Revenue"]
predictors = data[predictor_columns]
predictor_rows = list(predictors.itertuples(index=False, name=None))

def predictor_group_id(row):
    payload = json.dumps(
        {"columns": predictor_columns, "values": list(row)},
        ensure_ascii=False, allow_nan=False, separators=(",", ":"),
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

groups = pd.Series([predictor_group_id(row) for row in predictor_rows], index=data.index, name="predictor_group")
hash_to_tuple = {}
for group_id, row in zip(groups, predictor_rows):
    assert group_id not in hash_to_tuple or hash_to_tuple[group_id] == row, "Hash collision"
    hash_to_tuple[group_id] = row

group_sizes = groups.value_counts()
conflicting = data["Revenue"].groupby(groups).nunique().gt(1)
assert int(data.duplicated().sum()) == 125
assert int(data.duplicated(keep=False).sum()) == 201
assert int(predictors.duplicated().sum()) == 125
assert int(predictors.duplicated(keep=False).sum()) == 201
assert not conflicting.any()
assert groups.nunique() == 12205
assert int(group_sizes.gt(1).sum()) == 76
assert groups.duplicated().equals(predictors.duplicated())
# Label and row order do not enter the function: reversed records reverse IDs.
assert [predictor_group_id(row) for row in reversed(predictor_rows)] == groups.iloc[::-1].tolist()
print(f"Full-row repeats beyond first: {int(data.duplicated().sum())}")
print(f"Rows in repeated full-row groups: {int(data.duplicated(keep=False).sum())}")
print(f"Predictor-only repeats beyond first: {int(predictors.duplicated().sum())}")
print(f"Rows in repeated predictor groups: {int(predictors.duplicated(keep=False).sum())}")
print(f"Predictor groups: {groups.nunique()}; repeated groups: {int(group_sizes.gt(1).sum())}")
print(f"Predictor groups with conflicting Revenue labels: {int(conflicting.sum())}")
print("Group size -> number of groups:")
print(group_sizes.value_counts().sort_index().to_string())


Full-row repeats beyond first: 125
Rows in repeated full-row groups: 201
Predictor-only repeats beyond first: 125
Rows in repeated predictor groups: 201
Predictor groups: 12205; repeated groups: 76
Predictor groups with conflicting Revenue labels: 0
Group size -> number of groups:
count
1     12129
2        54
3        14
4         3
5         1
6         1
7         2
14        1


## Categories and calendar coverage

Frequency tables cover all observed levels, including small categories.
For this audit, “rare” means fewer than 20 records (a descriptive flag, not a
preprocessing threshold). Do not merge levels based on full-source frequencies.


In [4]:
frequency_columns = categorical_columns + ["Weekend", "SpecialDay"]
for column in frequency_columns:
    print(f"\n{column}:")
    print(data[column].value_counts(dropna=False).sort_index().to_string())
calendar_months = ["Jan", "Feb", "Mar", "Apr", "May", "June", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
missing_months = [month for month in calendar_months if month not in set(data["Month"])]
assert missing_months == ["Jan", "Apr"]
print(f"\nMissing calendar months: {missing_months}")
print("\nRare nominal levels (<20 rows):")
for column in categorical_columns:
    counts = data[column].value_counts()
    rare = counts[counts < 20].sort_index()
    if not rare.empty:
        print(f"{column}: {rare.to_dict()}")



Month:
Month
Aug      433
Dec     1727
Feb      184
Jul      432
June     288
Mar     1907
May     3364
Nov     2998
Oct      549
Sep      448

OperatingSystems:
OperatingSystems
1    2585
2    6601
3    2555
4     478
5       6
6      19
7       7
8      79

Browser:
Browser
1     2462
2     7961
3      105
4      736
5      467
6      174
7       49
8      135
9        1
10     163
11       6
12      10
13      61

Region:
Region
1    4780
2    1136
3    2403
4    1182
5     318
6     805
7     761
8     434
9     511

TrafficType:
TrafficType
1     2451
2     3913
3     2052
4     1069
5      260
6      444
7       40
8      343
9       42
10     450
11     247
12       1
13     738
14      13
15      38
16       3
17       1
18      10
19      17
20     198

VisitorType:
VisitorType
New_Visitor           1694
Other                   85
Returning_Visitor    10551

Weekend:
Weekend
False    9462
True     2868

SpecialDay:
SpecialDay
0.0    11079
0.2      178
0.4      243
0.6      35

## Skew, ranges and suspicious combinations

All numerical features are nonnegative. Rate features are stored as fractions;
their observed 0.2 ceiling is not established as a universal valid maximum.
Flag long tails and count/time combinations for later investigation, without
deleting or capping records. UCI does not specify time units or PageValues currency.


In [5]:
numeric = data[numeric_columns]
assert (numeric >= 0).all().all()
assert data[["BounceRates", "ExitRates", "SpecialDay"]].le(1).all().all()
summary = numeric.quantile([0, 0.5, 0.95, 0.99, 1]).T
summary.columns = ["min", "median", "p95", "p99", "max"]
summary["zero_count"] = numeric.eq(0).sum()
summary["skew"] = numeric.skew()
print(summary.to_string(float_format=lambda value: f"{value:.6f}"))
print("\nCount/time combinations:")
for count_column, duration_column in [
    ("Administrative", "Administrative_Duration"),
    ("Informational", "Informational_Duration"),
    ("ProductRelated", "ProductRelated_Duration"),
]:
    print(f"{count_column}: count=0/time>0: "
          f"{int(((data[count_column] == 0) & (data[duration_column] > 0)).sum())}; "
          f"count>0/time=0: {int(((data[count_column] > 0) & (data[duration_column] == 0)).sum())}")
for column in ["BounceRates", "ExitRates"]:
    print(f"{column} at 0.2: {int(data[column].eq(0.2).sum())}")
assert source.read_bytes() == source_bytes, "Source bytes changed during audit"
print("\nAudit passed; original CSV bytes unchanged. All 12,330 records retained.")


                             min     median         p95         p99          max  zero_count     skew
Administrative          0.000000   1.000000    9.000000   14.000000    27.000000        5768 1.960357
Administrative_Duration 0.000000   7.500000  348.266369  830.587041  3398.750000        5903 5.615719
Informational           0.000000   0.000000    3.000000    6.000000    24.000000        9699 4.036464
Informational_Duration  0.000000   0.000000  195.000000  716.390000  2549.375000        9925 7.579185
ProductRelated          0.000000  18.000000  109.000000  221.000000   705.000000          38 4.341516
ProductRelated_Duration 0.000000 598.936905 4300.289077 8701.142697 63973.522230         755 7.263228
BounceRates             0.000000   0.003112    0.200000    0.200000     0.200000        5518 2.947855
ExitRates               0.000000   0.025156    0.200000    0.200000     0.200000          76 2.148789
PageValues              0.000000   0.000000   38.160528   85.498490   361.763742  